# MangoScan Model 1 - Variety Classifier

Trains a 6-class YOLOv8-cls model on Colab.

**Before you start:** set Runtime -> Change runtime type -> **T4 GPU**.

Your dataset must be in Google Drive at
`MyDrive/MangoScan/data/raw/<Class_Name>/<fruit_id>_v<n>.jpg`.
See `docs/01-dataset-sourcing-guide.md`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
RAW = pathlib.Path('/content/drive/MyDrive/MangoScan/data/raw')
assert RAW.is_dir(), f'Not found: {RAW}. Upload your dataset there first.'
for d in sorted(RAW.iterdir()):
    if d.is_dir():
        print(f'{d.name:<14} {len(list(d.glob("*")))} files')

In [ ]:
!pip install -q ultralytics
!git clone -q https://github.com/mangoscan-a11y/MangoScan.git /content/MangoScan || true
%cd /content/MangoScan/MyTasks/model1_variety
!ls

## 1. Prepare the dataset

Splits by **physical fruit**, never by image - five views of one mango always
land in the same split. Splitting by image would inflate your accuracy and
invalidate the whole evaluation.

In [ ]:
import pathlib

cfg_path = pathlib.Path('configs/dataset.toml')
text = cfg_path.read_text()
text = text.replace('raw_dir = "data/raw"',
                    'raw_dir = "/content/drive/MyDrive/MangoScan/data/raw"')
text = text.replace('dataset_dir = "data/dataset"',
                    'dataset_dir = "/content/dataset"')
cfg_path.write_text(text)
print(text)

In [ ]:
!python scripts/01_prepare_dataset.py --config configs/dataset.toml

## 2. Train

~100 epochs on a T4 takes roughly 15-40 minutes for a few thousand images.
Watch `val/top1_acc`; early stopping fires after 20 epochs without
improvement.

In [ ]:
!python scripts/02_train.py --config configs/dataset.toml

## 3. Evaluate

Reports per-image accuracy, 5-view aggregated accuracy, the gain from
multi-view fusion, and Expected Calibration Error.

**ECE is the one to watch** - Model 5 gates routing on this model's
confidence, so an overconfident classifier misroutes fruit.

In [ ]:
!python scripts/03_evaluate.py --config configs/dataset.toml \
    --weights runs/model1_variety/weights/best.pt

In [ ]:
import csv, json, pathlib
import matplotlib.pyplot as plt

out = pathlib.Path('runs/model1_variety/weights')
report = json.loads((out / 'report.json').read_text())
classes = report['classes']

rows = list(csv.reader((out / 'confusion_matrix.csv').open()))
matrix = [[int(v) for v in r[1:]] for r in rows[1:]]

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(matrix, cmap='Blues')
ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f"Multi-view confusion - top-1 {report['multiview']['top1_accuracy']:.3f}")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, matrix[i][j], ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.savefig(out / 'confusion_matrix.png', dpi=150); plt.show()

bins = list(csv.DictReader((out / 'reliability.csv').open()))
xs = [float(b['avg_confidence']) for b in bins if int(b['count'])]
ys = [float(b['accuracy']) for b in bins if int(b['count'])]
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], '--', color='grey', label='perfect calibration')
ax.plot(xs, ys, 'o-', label='model')
ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
ax.set_title(f"Reliability - ECE {report['multiview']['ece']:.4f}")
ax.legend(); plt.tight_layout()
plt.savefig(out / 'reliability.png', dpi=150); plt.show()

## 4. Export

In [ ]:
!python scripts/04_export.py --config configs/dataset.toml \
    --weights runs/model1_variety/weights/best.pt

## 5. Save results back to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/MangoScan/runs
!cp -r runs/model1_variety /content/drive/MyDrive/MangoScan/runs/
print('Saved to MyDrive/MangoScan/runs/model1_variety')